### In this notebook we will perform the word embedding & topic modeling & Cosine Similarity

***we merged the three chapters to perform the topic modeling, in order to perform cosine similarity to select which chapter the new input should go with.***

In [7]:
!pip install PyMuPDF


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 67.9 MB/s eta 0:00:00
ERROR: Operation cancelled by user


In [1]:
!pip install numpy==1.24.4 scipy==1.11.4 gensim==4.3.1 spacy==3.5.3 thinc==8.1.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 1.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of spacy to determine which version is compatible with other requirements. This could take a while.
ERROR: Cannot install spacy==3.5.3 and thinc==8.1.7 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested thinc==8.1.7
    spacy 3.5.3 depends on thinc<8.2.0 and >=8.1.8

To fix this you could try to:
1. loosen the range of package versions you've specified
2. remove package versions to allow pip to attempt to solve the dependency conflict

ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts


In [2]:
import pandas as pd
import numpy as np
import pickle

# gensim
from gensim import corpora, models, similarities, matutils

# sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import NMF

### Read the data and pickle file

In [8]:
df02 = pd.read_csv('/content/all_chapters_3_rows.csv')

In [9]:
# reading the stop words list with pickle
with open ('stop_words.ob', 'rb') as fp:
    stop_words = pickle.load(fp)

In [10]:
df02.columns

Index(['string_values', 'Ch_No'], dtype='object')

In [11]:
# Declare a list that is to be converted into a column
ch_no = ['ear_nose', 'musculoskeletal', 'respiratory']

# Using 'ch_no' as the column name
# and equating it to the list
df02['Ch_No'] = ch_no

In [12]:
df02

,string_values,Ch_No
0,"13\nEar, Nose, and Throat Disorders\nIntroduct...",ear_nose
1,6\nMusculoskeletal Disorders\nIntroduction\nA ...,musculoskeletal
2,2\nRespiratory Disorders\nIntroduction\nThe re...,respiratory


### Word Embedding

In [13]:
df02['string_values']

,string_values
0,"13\nEar, Nose, and Throat Disorders\nIntroduct..."
1,6\nMusculoskeletal Disorders\nIntroduction\nA ...
2,2\nRespiratory Disorders\nIntroduction\nThe re...


In [14]:
# Create a CountVectorizer for parsing/counting words
count_vectorizer = CountVectorizer(stop_words=stop_words)

doc_word_cv = count_vectorizer.fit_transform(df02['string_values'])

/usr/local/lib/python3.11/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['00050', '10', '17028pleural', '2015', '2017', '297664', '45', '5674', '7faisy', 'abusei', 'afp', 'article', 'au', 'august', 'bounding', 'com', 'combined', 'cov', 'db', 'emedicine', 'epiglottis', 'ferruginous', 'forms', 'fromhttps', 'hammer', 'https', 'immobilization', 'infiltrates', 'iron', 'lamb', 'leg', 'mechanicalventilationviraladenovirus', 'medscape', 'midline', 'mortality', 'near', 'org', 'overviewnational', 'pain2respiratory', 'purulent', 'racgp', 'replacement', 'riskpatients', 'sars', 'stenosis', 'swool', 'tidalvolume', 'to7', 'toe', 'treceive', 'turkthoracj', 'urti', 'vasodilation', 'www'] not in stop_words.
  warnings.warn(


In [15]:
pd.DataFrame(doc_word_cv.toarray(), index=df02['Ch_No'], columns = count_vectorizer.get_feature_names_out()).head()

,000,0000000000000477,0000515997,0000531121,00016489,00050,001,0019,01,014,...,zealand,zheng,zinc,zones,zoster,zygote,µl,µm,μl,ඈ2
Ch_No,,,,,,,,,,,,,,,,,,,,,
ear_nose,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
musculoskeletal,2,1,0,0,0,0,1,0,0,0,...,1,0,0,0,0,1,0,0,1,0
respiratory,5,0,1,1,1,1,0,2,2,1,...,1,1,1,6,0,0,1,5,0,3


In [17]:
# Create a TfidfVectorizer for parsing/counting words
tfidf = TfidfVectorizer(stop_words=stop_words)

doc_word_tfidf = tfidf.fit_transform(df02['string_values'])

In [18]:
pd.DataFrame(doc_word_tfidf.toarray(), index=df02['Ch_No'], columns = tfidf.get_feature_names_out()).head()

,000,0000000000000477,0000515997,0000531121,00016489,00050,001,0019,01,014,...,zealand,zheng,zinc,zones,zoster,zygote,µl,µm,μl,ඈ2
Ch_No,,,,,,,,,,,,,,,,,,,,,
ear_nose,0.013106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.003698,0.000000,0.000000,0.000000,0.000000,0.00000
musculoskeletal,0.004468,0.003782,0.000000,0.000000,0.000000,0.000000,0.003782,0.000000,0.000000,0.000000,...,0.002877,0.000000,0.000000,0.000000,0.000000,0.003782,0.000000,0.000000,0.003782,0.00000
respiratory,0.010680,0.000000,0.003617,0.003617,0.003617,0.003617,0.000000,0.007233,0.007233,0.003617,...,0.002750,0.003617,0.003617,0.021699,0.000000,0.000000,0.003617,0.018083,0.000000,0.01085


### Topic Modeling: **LDA**

In [19]:
# Convert sparse matrix of counts to a gensim corpus
corpus = matutils.Sparse2Corpus(doc_word_cv)

In [20]:
id2word = dict((v, k) for k, v in count_vectorizer.vocabulary_.items())

In [21]:
# Create lda model (equivalent to "fit" in sklearn)
lda = models.LdaModel(corpus=corpus, num_topics=3, id2word=id2word, passes=5)

In [22]:
lda.print_topics(3)

[(0,
  '0.748*"0000000000000477" + 0.002*"000" + 0.001*"0000515997" + 0.000*"patency" + 0.000*"patch" + 0.000*"patches" + 0.000*"passively" + 0.000*"patent" + 0.000*"pathogens" + 0.000*"pathogen"'),
 (1,
  '0.623*"0000515997" + 0.262*"000" + 0.004*"0000000000000477" + 0.000*"patency" + 0.000*"patch" + 0.000*"patches" + 0.000*"passively" + 0.000*"patent" + 0.000*"pathogens" + 0.000*"pathogen"'),
 (2,
  '0.000*"0000515997" + 0.000*"0000000000000477" + 0.000*"000" + 0.000*"patency" + 0.000*"patch" + 0.000*"patches" + 0.000*"passively" + 0.000*"patent" + 0.000*"pathogens" + 0.000*"pathogen"')]

### Performing CorEx:

In [25]:
!pip install corextopic
from corextopic import corextopic as ct
from corextopic import vis_topic as vt

words = list(np.asarray(count_vectorizer.get_feature_names_out()))



In [26]:
topic_model = ct.Corex(n_hidden=3, words=words, seed=1)
topic_model.fit(doc_word_cv, words=words, docs=df02['string_values'])

In [27]:
topics = topic_model.get_topics()
for n,topic in enumerate(topics):
    topic_words,_,_ = zip(*topic)
    print('{}: '.format(n) + ','.join(topic_words))

0: inclusion,mucoid,moxifloxacin,mountain,mottled,mother,mosenifar,morphine,morbidity,moon
1: 0000000000000477,myopathies,myoglobin,myofibrils,myocardium,myelopathy,myelomeningocele,myeloma,myelography,myelogram
2: lavage,researchers,enzyme,enzymes,replacement,replaced,replace,equals,resistance,erosion


### Topic Modeling: LSA

In [28]:
lsa = TruncatedSVD(3)
doc_topic = lsa.fit_transform(doc_word_cv)
print(lsa.explained_variance_ratio_)

[0.01076154 0.5123585  0.47687995]


In [29]:
topic_word = pd.DataFrame(lsa.components_.round(3),
             index = ['component'+str(i) for i in range(3)],
             columns = count_vectorizer.get_feature_names_out())

print(topic_word)

              000  0000000000000477  0000515997  0000531121  00016489  00050  \
component0  0.015             0.001       0.001       0.001     0.001  0.001   
component1  0.009            -0.002      -0.001      -0.001    -0.001 -0.001   
component2 -0.004             0.002      -0.003      -0.003    -0.003 -0.003   

              001   0019     01    014  ...  zealand  zheng   zinc  zones  \
component0  0.001  0.003  0.003  0.001  ...    0.002  0.001  0.001  0.008   
component1 -0.002 -0.002 -0.002 -0.001  ...   -0.002 -0.001 -0.001 -0.005   
component2  0.002 -0.005 -0.005 -0.003  ...   -0.000 -0.003 -0.003 -0.015   

            zoster  zygote     µl     µm     μl     ඈ2  
component0   0.001   0.001  0.001  0.006  0.001  0.004  
component1   0.003  -0.002 -0.001 -0.004 -0.002 -0.002  
component2   0.001   0.002 -0.003 -0.013  0.002 -0.008  

[3 rows x 6012 columns]


In [30]:
tem_list = []
def display_topics(model, feature_names, no_top_words, topic_names=None):

    for ix, topic in enumerate(model.components_):
        inner_tem_list = []

        if not topic_names or not topic_names[ix]:
            print("\nTopic ", ix)
        else:
            print("\nTopic: '",topic_names[ix],"'")

        print(", ".join([feature_names[i]
                        for i in topic.argsort()[:-no_top_words - 1:-1]]))
        inner_tem_list.append(", ".join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))
        tem_list.append(inner_tem_list)

In [31]:
result1 = display_topics(lsa, count_vectorizer.get_feature_names_out(), 20)


Topic  0
symptoms, causes, signs, patients, bone, ear, complications, otitis, results, muscle, hearing, ventilation, oxygen, 10, changes, breathing, considerations, children, hours, airway

Topic  1
ear, otitis, hearing, media, sinusitis, nose, bleeding, throat, externa, membrane, canal, septum, polyps, speech, obstruction, voice, mouth, cord, packing, sinus

Topic  2
bone, muscle, joints, ear, hip, bones, deformity, traction, kyphosis, hearing, otitis, disk, exercises, brace, gout, cartilage, osteoporosis, muscles, causes, torticollis


In [32]:
tem_list
final_dic = {}
final_dic["Bone"] = tem_list[0]
final_dic["Ear"] = tem_list[1]
final_dic["Breathing"] = tem_list[2]

In [33]:
final_dic

{'Bone': ['symptoms, causes, signs, patients, bone, ear, complications, otitis, results, muscle, hearing, ventilation, oxygen, 10, changes, breathing, considerations, children, hours, airway'],
 'Ear': ['ear, otitis, hearing, media, sinusitis, nose, bleeding, throat, externa, membrane, canal, septum, polyps, speech, obstruction, voice, mouth, cord, packing, sinus'],
 'Breathing': ['bone, muscle, joints, ear, hip, bones, deformity, traction, kyphosis, hearing, otitis, disk, exercises, brace, gout, cartilage, osteoporosis, muscles, causes, torticollis']}

In [34]:
tem_df = pd.DataFrame.from_dict(final_dic, orient ='index')
tem_df

,0
Bone,"symptoms, causes, signs, patients, bone, ear, ..."
Ear,"ear, otitis, hearing, media, sinusitis, nose, ..."
Breathing,"bone, muscle, joints, ear, hip, bones, deformi..."


In [35]:
# Declare a list that is to be converted into a column
d_name = ['musculoskeletal', 'ear_nose', 'respiratory']

# Using 'ch_no' as the column name
# and equating it to the list
tem_df['D_Name'] = d_name

In [36]:
tem_df.columns

Index([0, 'D_Name'], dtype='object')

In [37]:
tem_df = tem_df.rename(columns={0: 'Description'})
tem_df

,Description,D_Name
Bone,"symptoms, causes, signs, patients, bone, ear, ...",musculoskeletal
Ear,"ear, otitis, hearing, media, sinusitis, nose, ...",ear_nose
Breathing,"bone, muscle, joints, ear, hip, bones, deformi...",respiratory


In [38]:
tem_df.to_csv('diseases_with_description.csv', index=False)